In [ ]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
  from google.colab import drive

  drive.mount('/content/drive') # load google drive
  os.chdir('/content/drive/My Drive/Thesis_Repository/Final_Google_Drive') # change directory to the current working directory

Mounted at /content/drive


## 1. Settings

In [ ]:
import json
import re
import random
import os
from pathlib import Path

import yaml

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    matthews_corrcoef
)

from transformers import (
    AutoTokenizer,
    # AutoModelForSequenceClassification,
    AutoModelForMaskedLM, # for maskd language model training
    # DataCollatorWithPadding,
    DataCollatorForLanguageModeling, # for masked language model training
    TrainingArguments,
    Trainer,
)

PRETRAINING_CONFIG_PATH = Path("./configs/further_pretraining.yaml")
MATHTUTOR_EEDI_TUTOR_ONLY_DATA_PATH = Path("./data/pretraining_data/mathtutormr_eedi__tutor_turn_only.csv")
CHECKPOINT_OUT_DIR = "./further_pretraining_model/further_pretraining__tutor_turn_only/checkpoints"
MODEL_OUT_DIR = "./further_pretraining_model/further_pretraining__tutor_turn_only/best_model"

MODEL_ID = "FacebookAI/roberta-base"
SEED = 42


# Set Seed
def set_all_seeds(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.mps.manual_seed(seed)
    except Exception:
        pass

set_all_seeds(SEED)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## 2. Load dataset

In [ ]:
df = pd.read_csv(MATHTUTOR_EEDI_TUTOR_ONLY_DATA_PATH)

In [ ]:
df.head()

,text
0,simplification?
1,how about combining all of those constants fir...
2,that doesnt sound right thonk i think 12 3 is ...
3,that cubic would not be that hard to factor.
4,that -9 will disappear you didn't differentiat...


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 944641 entries, 0 to 944640
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    944641 non-null  object
dtypes: object(1)
memory usage: 7.2+ MB


## 3. Train, Validation, Test split

The unlabeled corpus was split into training, validation, and test sets using a 98:1:1 ratio. Since the purpose of the corpus was continued masked-language-model pretraining rather than supervised evaluation, most of the data was retained for training. The validation and test sets still contained approximately 1,100 dialogue samples each, providing a sufficiently large number of token-level MLM predictions for monitoring validation loss and reporting held-out MLM performance.

The purpose of further pretraining is to improve the classification performance on the downstream task ( Don't Stop Pretraining : Adapt Language Models to Domain and Tasks)

The performance is evaluated on the downstream classification task.

In [ ]:
# 99 : 1

train_df, val_df = train_test_split(
    df,
    test_size=0.01,
    random_state=SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)


In [ ]:
print("[Split sizes]")
print("train:", len(train_df))
print("val:", len(val_df))

[Split sizes]
train: 935194
val: 9447


## 3. Batch Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID,  truncation=True, max_length=512, special_tokens=True)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        add_special_tokens=True, # include <s> and </s> tokens in the beginning and end of the text
    )


def build_dataset(split_data) -> Dataset:
    ds = Dataset.from_pandas(
        split_data,
        preserve_index=False,
    )

    ds = ds.map(
        tokenize_batch,
        batched=True,
        remove_columns=["text"],
    )

    return ds

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
from collections import Counter

# 결측치(NaN)가 있는 행을 제거
# train_df = train_df.dropna(subset=['text'])
# val_df = val_df.dropna(subset=['text'])

train_ds = build_dataset(train_df)
val_ds = build_dataset(val_df)
# test_ds = build_dataset(test_df)

print("\n[Dataset columns]")
print("train:", train_ds.column_names)
print("val:", val_ds.column_names)
# print("test:", test_ds.column_names)

print("\n[One tokenized example]")
print(train_ds[0].keys())
print(train_ds[0]['input_ids'])
print(tokenizer.decode(train_ds[0]['input_ids'], skip_special_tokens=False))


Map:   0%|          | 0/935194 [00:00<?, ? examples/s]

Map:   0%|          | 0/9447 [00:00<?, ? examples/s]


[Dataset columns]
train: ['input_ids', 'attention_mask']
val: ['input_ids', 'attention_mask']

[One tokenized example]
dict_keys(['input_ids', 'attention_mask'])
[0, 118, 524, 259, 7, 1457, 1649, 127, 5274, 13, 10, 4, 939, 342, 114, 5, 364, 118, 16, 1130, 172, 5, 3816, 20576, 1423, 23, 143, 477, 40, 8065, 741, 4, 114, 5, 1370, 181, 16, 8065, 172, 5, 3816, 20576, 23, 143, 477, 15, 23480, 40, 1130, 740, 4, 939, 342, 190, 114, 5, 5933, 16, 1130, 6, 53, 24, 1979, 75, 3327, 5, 3816, 20576, 23, 143, 477, 3023, 227, 5, 1370, 8, 5, 1461, 5933, 9, 5, 23480, 4, 35755, 35755, 35755, 313, 1268, 2854, 19, 127, 5274, 116, 35755, 6661, 116, 939, 206, 14, 114, 1615, 3488, 172, 181, 231, 523, 74, 7280, 3891, 111, 642, 231, 1615, 74, 712, 8, 1423, 16, 35563, 7, 111, 642, 231, 1615, 98, 24, 197, 712, 6, 235, 116, 67, 6, 47, 218, 75, 192, 457, 5367, 18, 244, 4238, 358, 183, 734, 939, 206, 14, 18, 235, 8, 13, 42, 734, 939, 33, 117, 1114, 141, 5, 5933, 8561, 1423, 939, 218, 75, 192, 143, 20631, 13, 5, 5933

## 4. Define a model

In [ ]:
# define a model for masked language modeling
model = AutoModelForMaskedLM.from_pretrained(MODEL_ID)

model.to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): 

## 5. Define a trainer

In [ ]:
# define data_collator for masked language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15, # masking proportion
)

In [ ]:
with PRETRAINING_CONFIG_PATH.open("r", encoding="utf-8") as f:
  config = yaml.safe_load(f)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_OUT_DIR,
    seed=SEED,
    **config["training_args"],
)


trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## 6. Train the model

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.206092,2.143649
2,2.115065,2.006385
3,1.975795,1.960790


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.decoder.weight', 'lm_head.decoder.bias'].


TrainOutput(global_step=43839, training_loss=2.1897138183429465, metrics={'train_runtime': 10111.8186, 'train_samples_per_second': 277.456, 'train_steps_per_second': 4.335, 'total_flos': 3.0573116257246714e+17, 'train_loss': 2.1897138183429465, 'epoch': 3.0})

7. Save the model

In [ ]:
trainer.save_model(MODEL_OUT_DIR)
tokenizer.save_pretrained(MODEL_OUT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./further_pretraining_model/further_pretrainng__tutor_turn_only/best_model/tokenizer_config.json',
 './further_pretraining_model/further_pretrainng__tutor_turn_only/best_model/tokenizer.json')

In [ ]:
from google.colab import runtime
runtime.unassign()